# Hantaan virus

In [127]:
import pandas as pd
import re

In [128]:
hantaan = pd.read_csv('../data/raw/hantaan_all.csv')
pd.set_option('display.max_colwidth', None)

In [129]:
hantaan.shape

(2342, 28)

In [130]:
nan_segments = hantaan[hantaan['Segment'].isna()]

In [131]:
len(nan_segments)

989

In [132]:
display(nan_segments[['Accession', 'GenBank_Title']].head(20))

,Accession,GenBank_Title
3,QP547404.1,JP 2025097909-A/413: SARS-COV-2 IMMUNOGENIC COMPOSITIONS
4,QP746458.1,JP 2025525390-A/1876: CIRCULAR RNA ENCODING CHIMERIC ANTIGEN RECEPTORS TARGETING BCMA
5,QS054047.1,JP 2025084850-A/5: HANTAVIRUS ANTIGENIC COMPOSITION
6,QS054048.1,JP 2025084850-A/6: HANTAVIRUS ANTIGENIC COMPOSITION
7,QS054049.1,JP 2025084850-A/8: HANTAVIRUS ANTIGENIC COMPOSITION
8,QS054054.1,JP 2025084850-A/18: HANTAVIRUS ANTIGENIC COMPOSITION
9,QS054055.1,JP 2025084850-A/19: HANTAVIRUS ANTIGENIC COMPOSITION
10,QS054056.1,JP 2025084850-A/20: HANTAVIRUS ANTIGENIC COMPOSITION
11,QS054061.1,JP 2025084850-A/25: HANTAVIRUS ANTIGENIC COMPOSITION
12,QS054062.1,JP 2025084850-A/26: HANTAVIRUS ANTIGENIC COMPOSITION


Uklonjene su metodoloski radovi, patentne prijave, vakcine  i patenti za vakcine.
Uklanjamo instance koje u nazivu sadrze kompozija jer to podrazumeva mesavinu vestacki napravljenih supstanci i ne predstavljaju prirodni izolat virusa. Takodje su obrisane i instance koje sadrze rec 'CHIMERIC' jer to oznacava vestacki stvoren gen ili organizam.

In [135]:
trash_words = 'patent|METHODS|VACCINE|COMPOSITIONS|COMPOSITION|CHIMERIC'

In [136]:
hantaan = hantaan[
    ~hantaan['GenBank_Title'].str.contains(trash_words, case=False, na=False)
]

In [137]:
hantaan.shape

(2241, 28)

In [138]:
nan_segments = hantaan[hantaan['Segment'].isna()]

In [139]:
len(nan_segments)

889

In [140]:
display(nan_segments[['Accession', 'GenBank_Title']].head(20))

,Accession,GenBank_Title
15,PX394250.1,"Orthohantavirus hantanense strain CJ64 RNA-dependent RNA polymerase gene, partial cds"
16,PX394251.1,"Orthohantavirus hantanense strain GC176 RNA-dependent RNA polymerase gene, partial cds"
17,PX394252.1,"Orthohantavirus hantanense strain YD192 RNA-dependent RNA polymerase gene, partial cds"
18,PX394253.1,"Orthohantavirus hantanense strain YD194 RNA-dependent RNA polymerase gene, partial cds"
19,PX394254.1,"Orthohantavirus hantanense strain YD198 RNA-dependent RNA polymerase gene, partial cds"
42,PP951047.1,"Orthohantavirus hantanense isolate Ap15-21 nucleocapsid protein gene, complete cds"
43,PP951048.1,"Orthohantavirus hantanense isolate Ap15-22 nucleocapsid protein gene, complete cds"
44,PP951049.1,"Orthohantavirus hantanense isolate Ap15-23 nucleocapsid protein gene, complete cds"
45,PP951050.1,"Orthohantavirus hantanense isolate Ap18-3 nucleocapsid protein gene, complete cds"
46,PP951051.1,"Orthohantavirus hantanense isolate Aa18-104 nucleocapsid protein gene, complete cds"


In [141]:
is_nan_segment = hantaan['Segment'].isna()

In [142]:
hantaan.loc[ is_nan_segment &
    hantaan['GenBank_Title'].str.contains(
        'nucleocapsid|nucleoprotein|segment N|{s segment}|S protein gene',
        case=False,
        na=False,
        regex=True
    ),
    'Segment'
] = 'S'

hantaan.loc[ is_nan_segment &
    hantaan['GenBank_Title'].str.contains(
        'glycoprotein|M protein gene| M polyprotein|M RNA segment|mRNA fragment|M fragment|polyprotein',
        case=False,
        na=False
    ),
    'Segment'
] = 'M'

hantaan.loc[ is_nan_segment &
    hantaan['GenBank_Title'].str.contains(
        'RNA-dependent RNA polymerase|L segment|L protein gene|RNA-polymerase-like gene',
        case=False,
        na=False,
        regex=True
    ),
    'Segment'
] = 'L'

In [143]:
hantaan['Segment'].value_counts(dropna=False)

Segment
M      859
S      701
L      657
NaN     23
G2       1
Name: count, dtype: int64

In [144]:
tmp = hantaan[hantaan['Segment'].isna() | (hantaan['Segment'] == 'G2')]
display(tmp[['Accession', 'GenBank_Title', 'Segment']])

,Accession,GenBank_Title,Segment
127,PP211262.1,MAG: Orthohantavirus hantanense isolate Ni.confucianus_hanta_10,NaN
128,PP211263.1,MAG: Orthohantavirus hantanense isolate Ni.confucianus_hanta_11,NaN
129,PP211264.1,MAG: Orthohantavirus hantanense isolate Ni.confucianus_hanta_12,NaN
130,PP211265.1,MAG: Orthohantavirus hantanense isolate Ni.confucianus_hanta_13,NaN
131,PP211266.1,MAG: Orthohantavirus hantanense isolate Ni.confucianus_hanta_14,NaN
132,PP211267.1,MAG: Orthohantavirus hantanense isolate Ni.confucianus_hanta_15,NaN
194,PL092189.1,"KR 1020240003760-A/41: New regulatory elements for enhancing RNA stability or mRNA translation, ZCCHC2 interacting with the same, and use thereof",NaN
195,PL092376.1,"KR 1020240003761-A/41: A method of screening regulatory elements for enhancing mRNA translation, new regulatory elements according to the method, and use thereof",NaN
196,PL092563.1,"KR 1020240003762-A/41: A method of screening regulatory elements for enhancing RNA stability or mRNA translation, new regulatory elements according to the method, and use thereof",NaN
592,OF364387.1,KR 1020220029898-A/3: SEROLOGICAL DETECTION METHOD USING ANTIGEN OF HTNV,NaN


Prebacivanje G2 zapisa u M segment (G2 glikoprotein pripada M segmentu)

In [145]:
hantaan['Segment'] = hantaan['Segment'].replace({'G2': 'M'})

Uklanjanje preostalih NaN unosa (patenti, MAG-ovi, sintetičke sekvence)

In [146]:
hantaan = hantaan.dropna(subset=['Segment']).copy()

In [147]:
hantaan['Segment'].value_counts(dropna=False)

Segment
M    860
S    701
L    657
Name: count, dtype: int64

In [148]:
cols = ['Accession','Segment',  'Nuc_Completeness', 'Species']

In [149]:
all_sequences = hantaan[cols]

In [150]:
all_sequences.to_csv('../data/processed_sequences/all_sequences/hantaan_all.csv')

In [151]:
complete_sequences = hantaan[hantaan['Nuc_Completeness'] == 'complete'].copy()

In [152]:
complete_sequences = complete_sequences[cols]

In [153]:
complete_sequences.to_csv('../data/processed_sequences/complete_sequences/hantaan_complete.csv')